chroma

In [1]:
#chroma api #import packages
from chroma import api
api.register_key("e424a1b4a1604a3a8cc83f0792dc3253")
from chroma import Protein, Chroma
from chroma import Chroma, conditioners
import os
from Bio import PDB
import pandas as pd
import csv
import locale
locale.getpreferredencoding = lambda: "UTF-8"
from chroma import Chroma, Protein, conditioners
from chroma.models import graph_classifier, procap
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import torch

/usr/local/lib/python3.10/dist-packages/chroma/layers/graph.py:27: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [5]:
#design based on length 
chroma = Chroma()
length = 160  # @param {type:"slider", min:50, max:250, step:10}

protein, trajectories = chroma.sample(
    chain_lengths=[length], steps=200, full_output=True
)
protein.to_CIF("protein.cif")
print(protein)

Using cached data from /tmp/chroma_weights/90e339502ae6b372797414167ce5a632/weights.pt
Loaded from cache
Data saved to /tmp/chroma_weights/03a3a9af343ae74998768a2711c8b7ce/weights.pt
Loaded from cache


Integrating SDE:   0%|          | 0/200 [00:00<?, ?it/s]

Potts Sampling:   0%|          | 0/500 [00:00<?, ?it/s]

Sequential decoding:   0%|          | 0/160 [00:00<?, ?it/s]

Protein: system
> Chain A (160 residues)
TEELIKRAIEEGVEGVARLLQELLSDPETLAKAVAEIVKALTPNPEKTKVRLIIEDDPEEAVRRANQEAEKALKEDGENTIIVLVVTRSAELPEPLSDEVLKKTTFISIVLPVETPDGLRAAVQVQVNTDTPEGEKLKQTLEAIAAAYQAAVDRLREERR




In [6]:
chroma = Chroma()
conditioner = conditioners.SymmetryConditioner(G="C_3", num_chain_neighbors=2)
protein = chroma.sample(
    chain_lengths=[100],
    conditioner=conditioner,
    langevin_factor=8,
    inverse_temperature=8,
    sde_func="langevin",
    potts_symmetry_order=conditioner.potts_symmetry_order)

protein.to("sample-C3.cif")
print(protein)

Using cached data from /tmp/chroma_weights/90e339502ae6b372797414167ce5a632/weights.pt
Loaded from cache
Using cached data from /tmp/chroma_weights/03a3a9af343ae74998768a2711c8b7ce/weights.pt
Loaded from cache


Integrating SDE:   0%|          | 0/500 [00:00<?, ?it/s]

Potts Sampling:   0%|          | 0/500 [00:00<?, ?it/s]

Sequential decoding:   0%|          | 0/300 [00:00<?, ?it/s]

Protein: system
> Chain A (100 residues)
QKLFVQAGMGVVTEVVITASDDASKEFIEELIKALEDMGMEVKADLVPESEASKSRAVARGTMDGEELAAKIKEIVDELEKRENYKKKIHLSSVYFEHPL

> Chain B (100 residues)
QKLFVQAGMGVVTEVVITASDDASKEFIEELIKALEDMGMEVKADLVPESEASKSRAVARGTMDGEELAAKIKEIVDELEKRENYKKKIHLSSVYFEHPL

> Chain C (100 residues)
QKLFVQAGMGVVTEVVITASDDASKEFIEELIKALEDMGMEVKADLVPESEASKSRAVARGTMDGEELAAKIKEIVDELEKRENYKKKIHLSSVYFEHPL




In [ ]:
def letter_to_point_cloud(
    letter="G",
    width_pixels=35,
    
    #font=os.path.join(
    #    os.path.dirname(chroma.__path__[0]), "./LiberationSans-Regular.ttf"
    #),
    font = "/workspace/chroma/LiberationSans-Regular.ttf",
    depth_ratio=0.15,
    fontsize_ratio=1.2,
    stroke_width=1,
    margin=0.5,
    max_points=2000,
):
    """Build a point cloud from a letter"""
    depth = int(depth_ratio * width_pixels)
    fontsize = int(fontsize_ratio * width_pixels)

    font = ImageFont.truetype(font, fontsize)
    ascent, descent = font.getmetrics()
    text_width = font.getmask(letter).getbbox()[2]
    text_height = font.getmask(letter).getbbox()[3] + descent

    margin_width = int(text_width * margin)
    margin_height = int(text_height * margin)
    image_size = [text_width + margin_width, text_height + margin_height]

    image = Image.new("RGBA", image_size, (255, 255, 255))
    draw = ImageDraw.Draw(image)
    draw.text(
        (margin_width // 2, margin_height // 2),
        letter,
        (0, 0, 0),
        font=font,
        stroke_width=stroke_width,
        stroke_fill="black",
    )

    A = np.asarray(image).mean(-1)
    A = A < 100.0
    V = np.ones(list(A.shape[:2]) + [depth]) * A[:, :, None]
    X_point_cloud = np.stack(np.nonzero(V), 1)
    # Uniform dequantization
    X_point_cloud = X_point_cloud + np.random.rand(*X_point_cloud.shape)

    if max_points is not None and X_point_cloud.shape[0] > max_points:
        np.random.shuffle(X_point_cloud)
        X_point_cloud = X_point_cloud[:max_points, :]

    return X_point_cloud

In [ ]:
#design based on shape
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
chroma = Chroma()
character = "G"  # @param {type:"string"}
if len(character) > 1:
    character = character[:1]
    print(f"Keeping only first character ({character})!")
length = 1000  # @param {type:"slider", min:100, max:1500, step:100}

letter_point_cloud = letter_to_point_cloud(character)
conditioner = conditioners.ShapeConditioner(
    letter_point_cloud,
    chroma.backbone_network.noise_schedule,
    autoscale_num_residues=length,
).to(device)

shaped_protein, trajectories = chroma.sample(
    chain_lengths=[length], conditioner=conditioner, full_output=True
)

shaped_protein.to_CIF("G.cif")
print(shaped_protein)

In [ ]:
from chroma import api
api.register_key("e424a1b4a1604a3a8cc83f0792dc3253")
from chroma import Protein, Chroma
from chroma import Chroma, conditioners
from chroma.utility.chroma import plane_split_protein


protein = Protein('/tmp/6ykm.pdb')

X, C, _ = protein.to_XCS()
selection_string = "namesel infilling_selection"  # @param ['namesel infilling_selection', 'z > 16', '(resid 50) around 10'] {allow-input:true}
residues_to_design = plane_split_protein(X, C, protein, 0.5).nonzero()[:, 1].tolist()
protein.sys.save_selection(gti=residues_to_design, selname="infilling_selection")

try:
    conditioner = conditioners.SubstructureConditioner(
        protein, backbone_model=chroma.backbone_network, selection=selection_string
    ).to(device)
except Exception:
    print("Error initializing conditioner! Falling back to masking 50% of residues.")
    selection_string = "namesel infilling_selection"
    conditioner = conditioners.SubstructureConditioner(
        protein,
        backbone_model=chroma.backbone_network,
        selection=selection_string,
        rg=True,
    ).to(device)

infilled_protein, trajectories = chroma.sample(
    protein_init=protein,
    conditioner=conditioner,
    langevin_factor=4.0,
    langevin_isothermal=True,
    inverse_temperature=8.0,
    steps=500,
    sde_func="langevin",
    full_output=True,
)
protein.to("infilled_protein.pdb")

In [ ]:
# Redesign a Protein
from chroma import api
api.register_key("e424a1b4a1604a3a8cc83f0792dc3253")
from chroma import Protein, Chroma

chroma = Chroma()

protein = Protein('/tmp/6ykm.pdb') # PDB is fine too
protein = chroma.design(protein, design_selection="resid 20-50 around 5.0") #  5 angstrom bubble around indices 20-50

protein.to("6ykm_redesign.pdb")